# SmolVLM best-checkpoint inference from vision cache
Enable a Kaggle GPU and Internet. This notebook loads the server best checkpoint with canonical Front-to-Back cameras, FP32, batch 1. It does not train, extract images, publish datasets or collect attention yet.
Attach `huyhuy12334/cache-smolvlm` as Input if possible. kagglehub reads the existing dataset; the cache is not copied into working. Checkpoint/prepared downloads use public HF repositories. First validation scans cache finiteness in bounded chunks; later predictions reuse the session.


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from pathlib import Path
import subprocess, sys
REPO = Path('/kaggle/working/Log_Metrics')
BRANCH = 'smolvlm-attention'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/huyvanzzz/Log_Metrics.git', str(REPO)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
    if actual != BRANCH:
        raise RuntimeError(f'Expected {BRANCH}; existing checkout is {actual}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(REPO / 'smolvlm_inference/requirements.txt')], check=True)
sys.path.insert(0, str(REPO))
# If transformers/tokenizers were imported before installation, restart kernel then Run All.


In [ ]:
import torch, transformers, tokenizers
assert transformers.__version__ == '4.46.3'
assert tokenizers.__version__ == '0.20.4'
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before loading the model')
CHECKPOINT_REPO = 'huyvanzzz/smolvlm256M_base'
CHECKPOINT_REVISION = 'f9d4e9a4d704078f9e7f39f061d13602cf251c78'
CHECKPOINT_FILENAME = 'finetune_best.pt'
PREPARED_REPO = 'huyvanzzz/prepared_data'
PREPARED_REVISION = 'd2a6914f681dbb9148c4ac11e6fa92452010506e'
CACHE_KAGGLE_DATASET = 'huyhuy12334/cache-smolvlm'
# Optional: paste the mounted Input cache directory containing all three cache files.
CACHE_INPUT_DIR = ''
ARTIFACT_DIR = Path('/kaggle/working/smolvlm-inference')
ARTIFACT_DIR.mkdir(exist_ok=True)
print({'torch': torch.__version__, 'transformers': transformers.__version__,
       'gpu': torch.cuda.get_device_name(0), 'precision': 'fp32',
       'camera_order': ['Front', 'Front-Left', 'Front-Right', 'Back', 'Back-Left', 'Back-Right']})


In [ ]:
from huggingface_hub import hf_hub_download
import kagglehub
prepared_dir = ARTIFACT_DIR / 'prepared_data'
for filename in ('manifest.json', 'train.jsonl', 'val.jsonl', 'test.jsonl'):
    hf_hub_download(repo_id=PREPARED_REPO, repo_type='dataset', revision=PREPARED_REVISION,
                    filename=filename, local_dir=prepared_dir, token=False)
checkpoint_path = hf_hub_download(repo_id=CHECKPOINT_REPO, revision=CHECKPOINT_REVISION,
                                  filename=CHECKPOINT_FILENAME, token=False)
cache_root = Path(CACHE_INPUT_DIR) if CACHE_INPUT_DIR else Path(kagglehub.dataset_download(CACHE_KAGGLE_DATASET))
# Resolve a possible dataset packaging subdirectory without copying any features.
candidates = [p.parent for p in cache_root.rglob('features.float16.memmap')
              if (p.parent / 'index.json').is_file() and (p.parent / 'manifest.json').is_file()]
if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one complete feature cache under {cache_root}; found {len(candidates)}')
cache_dir = candidates[0]
print({'checkpoint': checkpoint_path, 'cache': str(cache_dir), 'prepared': str(prepared_dir)})


In [ ]:
from smolvlm_inference.session import InferenceSession
session = InferenceSession(checkpoint_path, cache_dir, prepared_dir, device='cuda:0')
print(session.describe())


In [ ]:
# Change only these two values to try another sample; keep the loaded session.
SPLIT = 'test'
SAMPLE_INDEX = 0
result = session.predict(SPLIT, SAMPLE_INDEX)
for key, value in result.items():
    print(f'{key}: {value}')
